# 03 Explore MART

- apre il primo mart dichiarato in `dataset.yml`
- mostra schema, preview e aggregazioni esplorative
- aiuta a collegare i mart alle domande civiche

In [ ]:
from pathlib import Path
import json
import shutil
import subprocess
import duckdb
import yaml

ROOT = Path('.').resolve()
DATASET_YML = (ROOT / 'dataset.yml').resolve() if (ROOT / 'dataset.yml').exists() else (ROOT / '..' / 'dataset.yml').resolve()
CFG = yaml.safe_load(DATASET_YML.read_text(encoding='utf-8'))
DATASET = CFG['dataset']['name']
YEARS = CFG['dataset']['years']
YEAR_INDEX = 0
YEAR = YEARS[YEAR_INDEX] if YEARS and 0 <= YEAR_INDEX < len(YEARS) else YEARS[0]
TABLES = CFG.get('mart', {}).get('tables', [])
TABLE_INDEX = 0
SELECTED_TABLE = TABLES[TABLE_INDEX] if TABLES and 0 <= TABLE_INDEX < len(TABLES) else (TABLES[0] if TABLES else {'name': 'mart_ok'})
TABLE_NAME = SELECTED_TABLE['name']
CLI_PREFIX = ['toolkit'] if shutil.which('toolkit') else ['py', '-m', 'toolkit.cli.app']
INSPECT_CMD = CLI_PREFIX + ['inspect', 'paths', '--config', str(DATASET_YML), '--year', str(YEAR), '--json']
INSPECT = json.loads(subprocess.run(INSPECT_CMD, capture_output=True, text=True, check=True).stdout)
MART_OUTPUTS = INSPECT['paths']['mart']['outputs']
MART_PATH = Path(MART_OUTPUTS[TABLE_INDEX]) if MART_OUTPUTS and 0 <= TABLE_INDEX < len(MART_OUTPUTS) else (Path(MART_OUTPUTS[0]) if MART_OUTPUTS else Path(INSPECT['paths']['mart']['dir']) / f'{TABLE_NAME}.parquet')
{'YEARS': YEARS, 'YEAR_INDEX': YEAR_INDEX, 'TABLES': [table['name'] for table in TABLES], 'TABLE_INDEX': TABLE_INDEX, 'TABLE_NAME': TABLE_NAME, 'MART_PATH': str(MART_PATH), 'INSPECT_CMD': INSPECT_CMD}

In [ ]:
con = duckdb.connect()
YEAR_COL = None
METRIC_COL = None

def choose_columns(schema_rows):
    year_col = next((row[0] for row in schema_rows if str(row[0]).lower() == 'year' or 'anno' in str(row[0]).lower()), None)
    numeric_rows = [row for row in schema_rows if any(token in str(row[1]).upper() for token in ['INT', 'DECIMAL', 'DOUBLE', 'FLOAT', 'REAL', 'BIGINT'])]
    metric_col = next((row[0] for row in numeric_rows if any(token in str(row[0]).lower() for token in ['value', 'tot', 'importo', 'ammontare', 'saldo', 'spese', 'entrate', 'pct', 'percent'])), None)
    if metric_col is None and numeric_rows:
        metric_col = numeric_rows[0][0]
    return year_col, metric_col

if MART_PATH.exists():
    schema_rows = con.execute(f"DESCRIBE SELECT * FROM read_parquet('{MART_PATH.as_posix()}')").fetchall()
    schema_df = con.execute(f"DESCRIBE SELECT * FROM read_parquet('{MART_PATH.as_posix()}')").df()
    preview_df = con.execute(f"SELECT * FROM read_parquet('{MART_PATH.as_posix()}') LIMIT 20").df()
    YEAR_COL, METRIC_COL = choose_columns(schema_rows)
    display(schema_df)
    display(preview_df)
    print({'YEAR_COL': YEAR_COL, 'METRIC_COL': METRIC_COL})
else:
    print('MART parquet not found. Run toolkit run mart --config dataset.yml first.')

In [ ]:
if MART_PATH.exists() and YEAR_COL and METRIC_COL:
    by_year = con.execute(
        f"SELECT {YEAR_COL} AS year_like, COUNT(*) AS rows, SUM({METRIC_COL}) AS metric_total FROM read_parquet('{MART_PATH.as_posix()}') GROUP BY 1 ORDER BY 1"
    ).df()
    display(by_year)
else:
    print('No year-like column or metric column detected.')